# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant-structured dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL as defined by FAIR principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")
print(f"Published: {metadata.date_published}")
print(f"Fields: Keywords={metadata.keywords}, Version={metadata.version}, License={metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All identities refer to their Croissant `@id`.

**List record set IDs and their field (column) IDs:**

In [ ]:
# Get available record sets and associated fields (`columns`)
record_sets = list(dataset.record_sets.keys())
print("Record Sets @ids in this dataset:")
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"  @id: {rs_id}\n    Name: {record_set.name}")
    # List fields (columns)
    print("    Field (column) @ids and names:")
    for field_id in record_set.fields:
        field = dataset.fields[field_id]
        col_str = f"    - @id: {field_id}, name: {getattr(field, 'name', '[no name]')}"
        print(col_str)
    print("-")
if not record_sets:
    print("No record sets were found in the Croissant metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

**Below, we load data for all available record sets and display the first few rows for each.**

In [ ]:
# If there are no record sets, dataset.records() will not yield results
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f"\nLoading records for record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print("  No records found.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping by categorical fields.

**In this example, we select a record set, choose a numeric field, filter, and normalize.**

In [ ]:
import numpy as np
"""
Choose a record set and IDs for a numeric field and grouping field.
Update these values as needed according to your dataset's field IDs and types.
"""
# Example initialization (replace with real IDs from the overview above if present)
if dataframes:
    # Choose the first available record set
    active_record_set = record_sets[0]
    df = dataframes[active_record_set]
    print(f"\nUsing record set @id: {active_record_set}")
    # Find a numeric field automatically
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Find a likely group field
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < min(10, len(df)//4):
            group_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric fields found.")
    else:
        # Filter and normalize
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group if applicable
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df)
        else:
            print("No suitable grouping field found.")
else:
    print("No data frames available; please check previous steps.")

## 5. Visualization
Visualize the distribution of the numeric field or comparison by a grouping field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if present
    if group_field_id:
        plt.figure(figsize=(9,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization not possible: No suitable numeric field available.")

## 6. Conclusion
Key observations and findings are dataset-specific and should be drawn from the summary statistics, distributions, and field definitions. 

With the `mlcroissant` library, you can reproducibly identify all data entities via their `@id`, ensuring FAIR and robust analytics. 

- Use the provided Croissant identifiers when referencing dataset components in your research or workflows.
- Proceed to model building, deeper EDA, or export as needed!
